# 한국어 데이터로 챗봇 만들기

In [1]:
import torch
print(torch.__version__)

2.7.1+cu118


1. 데이터 수집하기

In [4]:
!mkdir -p ~/work/transformer_chatbot/data/ && cd ~/work/transformer_chatbot/data/
!wget https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv

--2026-02-05 02:30:00--  https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv [following]
--2026-02-05 02:30:00--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘ChatbotData.csv’

ChatbotData.csv     100%[===================>] 868.99K  4.97MB/s    in 0.2s    

2026-02-05 02:30:01 (4.97 MB/s) - ‘ChatbotData.csv’ saved [889842/889842]



In [6]:
import pandas as pd

data = pd.read_csv('~/work/songys_chatbot/ChatbotData.csv')
print(data.head())

                 Q            A  label
0           12시 땡!   하루가 또 가네요.      0
1      1지망 학교 떨어졌어    위로해 드립니다.      0
2     3박4일 놀러가고 싶다  여행은 언제나 좋죠.      0
3  3박4일 정도 놀러가고 싶다  여행은 언제나 좋죠.      0
4          PPL 심하네   눈살이 찌푸려지죠.      0


2. 한국어 전처리

In [17]:
import re


def preprocess(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence


questions = [preprocess(q) for q in data['Q']]
answers = [preprocess(a) for a in data['A']]

In [35]:
import os
def preprocess(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence

# 1. CSV 로드
data = pd.read_csv('~/work/songys_chatbot/ChatbotData.csv')

# 2. Q/A 전처리
questions = [preprocess(q) for q in data['Q']]
answers   = [preprocess(a) for a in data['A']]

# 3. chatbot.txt 생성
path = os.path.expanduser('~/work/songys_chatbot/chatbot.txt')
with open(path, 'w', encoding='utf-8') as f:
    for q, a in zip(questions, answers):
        f.write(q + '\n')
        f.write(a + '\n')


3. SentencePiece tokenizer

In [12]:
import re

def preprocess_sentence(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence

In [25]:
import os

os.makedirs('tokenizer', exist_ok=True)


In [37]:
import sentencepiece as spm

path = '/home/jovyan/work/songys_chatbot/chatbot.txt'
spm.SentencePieceTrainer.train(
    input='/home/jovyan/work/songys_chatbot/chatbot.txt',
    model_prefix='tokenizer/spm',
    vocab_size=8000,
    model_type='bpe',
    pad_id=0,
    bos_id=1,
    eos_id=2,
    unk_id=3
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /home/jovyan/work/songys_chatbot/chatbot.txt
  input_format: 
  model_prefix: tokenizer/spm
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable

In [38]:
sp = spm.SentencePieceProcessor()
sp.load('/home/jovyan/work/AIFFEL_quest_rs/Exploration/Ex09/tokenizer/spm.model')

True

4. 데이터셋 구성

In [39]:
import torch
from torch.utils.data import Dataset


class ChatbotDataset(Dataset):
    def __init__(self, questions, answers, sp, max_len=40):
        self.q = questions
        self.a = answers
        self.sp = sp
        self.max_len = max_len
    
    
    def encode(self, sentence):
        tokens = [self.sp.bos_id()] + self.sp.encode_as_ids(sentence) + [self.sp.eos_id()]
        tokens = tokens[:self.max_len]
        tokens += [self.sp.pad_id()] * (self.max_len - len(tokens))
        return tokens
    
    
    def __len__(self):
        return len(self.q)
    
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.encode(self.q[idx])),
            torch.tensor(self.encode(self.a[idx]))
        )

5. 트랜스포머 모델

In [44]:
import torch.nn as nn


model = nn.Transformer(
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    batch_first = True
)


embedding = nn.Embedding(sp.get_piece_size(), 512)
fc_out = nn.Linear(512, sp.get_piece_size())

In [41]:
criterion = nn.CrossEntropyLoss(ignore_index=sp.pad_id())
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [45]:
def predict(sentence, max_len=40):
    model.eval()
    sentence = preprocess(sentence)
    
    
    enc = torch.tensor([
        sp.bos_id()] + sp.encode_as_ids(sentence) + [sp.eos_id()]
    ).unsqueeze(1)
    
    
    dec = torch.tensor([[sp.bos_id()]])
    
    
    for _ in range(max_len):
        out = model(enc, dec)
        token = out.argmax(-1)[-1].item()
        if token == sp.eos_id():
            break
        dec = torch.cat([dec, torch.tensor([[token]])], dim=1)
    
    
    return sp.decode_ids(dec.squeeze().tolist())

In [46]:
import torch
from torch.utils.data import Dataset, DataLoader

class ChatbotDataset(Dataset):
    def __init__(self, questions, answers, sp, max_len=40):
        self.questions = questions
        self.answers = answers
        self.sp = sp
        self.max_len = max_len

    def encode(self, sentence):
        tokens = [self.sp.bos_id()] + self.sp.encode_as_ids(sentence) + [self.sp.eos_id()]
        tokens = tokens[:self.max_len]
        tokens += [self.sp.pad_id()] * (self.max_len - len(tokens))
        return torch.tensor(tokens)

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        src = self.encode(self.questions[idx])
        tgt = self.encode(self.answers[idx])
        return src, tgt


In [47]:
def generate_square_subsequent_mask(sz):
    mask = torch.triu(torch.ones(sz, sz), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask


In [48]:
import torch.nn as nn

class TransformerChatbot(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=2048,
            dropout=0.1,
            batch_first=True
        )
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, tgt_mask):
        src = self.embedding(src)
        tgt = self.embedding(tgt)
        out = self.transformer(src, tgt, tgt_mask=tgt_mask)
        return self.fc_out(out)


In [49]:
import pandas as pd
import sentencepiece as spm
import re

def preprocess(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence

data = pd.read_csv('ChatbotData.csv')

questions = [preprocess(q) for q in data['Q']]
answers   = [preprocess(a) for a in data['A']]

sp = spm.SentencePieceProcessor()
sp.load('tokenizer/spm.model')

dataset = ChatbotDataset(questions, answers, sp)
loader = DataLoader(dataset, batch_size=64, shuffle=True)


In [50]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = TransformerChatbot(vocab_size=sp.get_piece_size()).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=sp.pad_id())
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        tgt_mask = generate_square_subsequent_mask(tgt_input.size(1)).to(device)

        optimizer.zero_grad()

        logits = model(src, tgt_input, tgt_mask)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Epoch {epoch+1}] Loss: {total_loss / len(loader):.4f}")


[Epoch 1] Loss: 6.0660
[Epoch 2] Loss: 5.4537
[Epoch 3] Loss: 5.1437
[Epoch 4] Loss: 4.8129
[Epoch 5] Loss: 4.4785
[Epoch 6] Loss: 4.1520
[Epoch 7] Loss: 3.8456
[Epoch 8] Loss: 3.5577
[Epoch 9] Loss: 3.2910
[Epoch 10] Loss: 3.0418


In [51]:
def decode_ids(ids, sp):
    tokens = []
    for i in ids:
        if i == sp.eos_id():
            break
        if i not in [sp.pad_id(), sp.bos_id()]:
            tokens.append(i)
    return sp.decode_ids(tokens)


In [56]:
def predict(model, sentence, sp, max_len=40):
    model.eval()

    # 입력 문장 전처리 & 인코딩
    sentence = preprocess(sentence)
    src_ids = [sp.bos_id()] + sp.encode_as_ids(sentence) + [sp.eos_id()]
    src_ids = src_ids[:max_len]
    src_ids += [sp.pad_id()] * (max_len - len(src_ids))

    src = torch.tensor(src_ids).unsqueeze(0).to(device)

    # 디코더 입력 (BOS부터 시작)
    tgt_ids = [sp.bos_id()]

    for _ in range(max_len):
        tgt = torch.tensor(tgt_ids).unsqueeze(0).to(device)

        tgt_mask = generate_square_subsequent_mask(tgt.size(1)).to(device)

        with torch.no_grad():
            output = model(src, tgt, tgt_mask)

        # 🔥 여기서부터 바뀜
        temperature = 0.8
        logits = output[0, -1] / temperature
        
        topk = 10
        values, indices = torch.topk(logits, topk)
        probs = torch.softmax(values, dim=-1)
        next_token = indices[torch.multinomial(probs, 1)].item()
        
        tgt_ids.append(next_token)
        
        if len(tgt_ids) > 3 and tgt_ids[-1] == tgt_ids[-2]:
            break
        if next_token == sp.eos_id():
            break

    return decode_ids(tgt_ids, sp)


In [ ]:
while True:
    question = input("👤 사용자: ")
    if question.lower() in ['quit', 'exit']:
        break

    answer = predict(model, question, sp)
    print("🤖 챗봇:", answer)


👤 사용자:  안녕하세요


🤖 챗봇: 잘 살 수 있어요 .


👤 사용자:  누구세요


🤖 챗봇: 좋은 곳으로 데려다 줄 거예요 .


회고

학습된 모델은 입력 문장에 대해 문법적으로 자연스러운 한국어 문장을 생성하였으나, 문맥을 고려한 대화 응답보다는 데이터 분포에 기반한 일반적인 답변을 주로 생성하였다.